# Phase 2: Fixed-MSFA Baseline

Goal: train a strong fixed-mask baseline and log paper-ready metrics.

Deliverables:
- `msfa_dataset_4x4.npz`
- `baseline_history.csv`
- `unet_baseline_best.pth`
- baseline qualitative figure


# Learnable MSFA Research Track

This notebook is part of the 10-day publication-oriented extension track.

## Run discipline
- Keep one experiment purpose per notebook.
- Save metrics and checkpoints to the configured project root.
- Do not silently change data splits or loss definitions across notebooks.
- When a result becomes final, export both numeric artifacts and a figure/table asset.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
except Exception as exc:
    print("Drive mount skipped:", exc)


In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path("/content/drive/MyDrive/Msa-Osp")
PATCH_PATH = PROJECT_ROOT / "dataset_patches.npz"
MSFA_DATA_PATH = PROJECT_ROOT / "msfa_dataset_4x4.npz"
HISTORY_PATH = PROJECT_ROOT / "baseline_history.csv"
CKPT_PATH = PROJECT_ROOT / "unet_baseline_best.pth"
FIG_PATH = PROJECT_ROOT / "baseline_example.png"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 2
EPOCHS = 40
LR = 1e-4
BASE = 32
BAND_COUNT = 16


In [ ]:
data = np.load(PATCH_PATH)
train_target = data["train"]
val_target = data["val"]

def create_fixed_mask(h, w, bands=BAND_COUNT):
    tile = np.arange(bands).reshape(4, 4)
    mask = np.zeros((h, w, bands), dtype=np.float32)
    for i in range(h):
        for j in range(w):
            mask[i, j, tile[i % 4, j % 4]] = 1.0
    return mask

mask = create_fixed_mask(train_target.shape[1], train_target.shape[2])

def apply_msfa(cubes, mask_3d):
    mosaiced = np.sum(cubes * mask_3d[None, ...], axis=-1)
    inputs = mosaiced[..., None] * mask_3d[None, ...]
    return inputs.astype(np.float32), cubes.astype(np.float32)

train_input, train_target = apply_msfa(train_target, mask)
val_input, val_target = apply_msfa(val_target, mask)

MSFA_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    MSFA_DATA_PATH,
    train_input=train_input,
    train_target=train_target,
    val_input=val_input,
    val_target=val_target,
)


In [ ]:
class MSFADataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.inputs[idx]).permute(2, 0, 1).float()
        y = torch.from_numpy(self.targets[idx]).permute(2, 0, 1).float()
        return x, y

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNet2D(nn.Module):
    def __init__(self, in_ch=16, out_ch=16, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 2, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.final = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.final(d1)

def compute_psnr(pred, target, eps=1e-8):
    mse = torch.mean((pred - target) ** 2)
    return 20 * torch.log10(1.0 / torch.sqrt(mse + eps))

def spectral_angle_mapper(pred, target, eps=1e-8):
    dot = torch.sum(pred * target, dim=1)
    pred_norm = torch.norm(pred, dim=1)
    target_norm = torch.norm(target, dim=1)
    cos_theta = torch.clamp(dot / (pred_norm * target_norm + eps), -1 + eps, 1 - eps)
    return torch.rad2deg(torch.acos(cos_theta)).mean()

def spectral_angle_loss(pred, target, eps=1e-8):
    dot = torch.sum(pred * target, dim=1)
    pred_norm = torch.norm(pred, dim=1)
    target_norm = torch.norm(target, dim=1)
    cos_theta = torch.clamp(dot / (pred_norm * target_norm + eps), -1 + eps, 1 - eps)
    return torch.acos(cos_theta).mean()

def compute_rgb_ssim(pred, target):
    if not HAS_SSIM:
        return float("nan")
    pred_np = pred.detach().cpu().numpy()
    target_np = target.detach().cpu().numpy()
    scores = []
    for i in range(pred_np.shape[0]):
        p = np.transpose(pred_np[i, [5, 10, 15]], (1, 2, 0))
        t = np.transpose(target_np[i, [5, 10, 15]], (1, 2, 0))
        data_range = max(float(t.max() - t.min()), 1e-8)
        scores.append(ssim_fn(t, p, data_range=data_range, channel_axis=2))
    return float(np.mean(scores))


In [ ]:
train_loader = DataLoader(MSFADataset(train_input, train_target), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(MSFADataset(val_input, val_target), batch_size=BATCH_SIZE, shuffle=False)

model = UNet2D(base=BASE).to(DEVICE)
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_psnr = -float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_psnr = 0.0
    val_sam = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x)
            val_psnr += compute_psnr(pred, y).item()
            val_sam += spectral_angle_mapper(pred, y).item()

    train_loss /= len(train_loader)
    val_psnr /= len(val_loader)
    val_sam /= len(val_loader)
    history.append({"epoch": epoch, "train_l1": train_loss, "val_psnr": val_psnr, "val_sam_deg": val_sam})
    print(f"Epoch {epoch:02d} | train L1 {train_loss:.4f} | val PSNR {val_psnr:.2f} dB | val SAM {val_sam:.2f} deg")

    if val_psnr > best_psnr:
        best_psnr = val_psnr
        CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_psnr": best_psnr,
            },
            CKPT_PATH,
        )

HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(HISTORY_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    writer.writeheader()
    writer.writerows(history)
